In [1]:
!pip install pandas
!pip install scikit-learn
!pip install pyarrow

In [2]:
!pip freeze | grep scikit-learn

scikit-learn==1.8.0


In [3]:
!python -V

Python 3.13.12


In [12]:
import pickle
import pandas as pd
import numpy as np

In [5]:

with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

/home/leonematt/miniconda/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/leonematt/miniconda/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
def get_data(year, month):
    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year}-{month:02d}.parquet'
    df = pd.read_parquet(url)
    return df

In [15]:
categorical = ['PULocationID', 'DOLocationID']

def engineer_data(df, year, month):
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()
    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
    
    return df


In [ ]:
df = get_data(2023, 3)
df = engineer_data(df, 2023, 3)

In [17]:

dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [18]:
print(np.std(y_pred))

6.310194748365501


In [28]:
df_result = pd.DataFrame({
    'ride_id': df['ride_id'],
    'predicted_duration': y_pred,
})

df_result.to_parquet(
    "results.parquet",
    engine='pyarrow',
    compression=None,
    index=False
)